In [2]:
%pip install "crewai[tools]"
%pip install -U google-genai


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import google.genai

print("Google GenAI SDK installed successfully")

Google GenAI SDK installed successfully


In [4]:
import os
from crewai import LLM
import asyncio
from crewai import Crew, Agent, Task
from dotenv import load_dotenv

load_dotenv(override=True)
print("Environment variables loaded.")

gemini_api_key = os.getenv("GEMINI_API_KEY")

gemini_llm = LLM(
    model="gemini/gemini-3.6-flash",  # Must include 'gemini/' prefix
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.0
)

Environment variables loaded.


# Tools

In [5]:
from crewai.tools import BaseTool

# Placeholder tool for fetching customer support data
class CustomerSupportDataTool(BaseTool):
    name: str = "Customer Support Data Fetcher"
    description: str = (
      "Fetches recent customer support interactions, tickets, and feedback. "
      "Returns a summary string.")

    def _run(self, argument: str) -> str:
        # In a real scenario, this would query a database or API.
        # For this example, return simulated data.
        print(f"--- Fetching data for query: {argument} ---")
        return (
            """Recent Support Data Summary:
- 50 tickets related to 'login issues'. High resolution time (avg 48h).
- 30 tickets about 'billing discrepancies'. Mostly resolved within 12h.
- 20 tickets on 'feature requests'. Often closed without resolution.
- Frequent feedback mentions 'confusing user interface' for password reset.
- High volume of calls related to 'account verification process'.
- Sentiment analysis shows growing frustration with 'login issues' resolution time.
- Support agent notes indicate difficulty reproducing 'login issues'."""
        )

support_data_tool = CustomerSupportDataTool()

# Agent

In [6]:
from crewai import Agent

# Agent 1: Data analyst
data_analyst = Agent(
    role='Customer Support Data Analyst',
    goal='Analyze customer support data to identify trends, recurring issues, and key pain points.',
    backstory=(
        """You are an expert data analyst specializing in customer support operations.
        Your strength lies in identifying patterns and quantifying problems from raw support data."""
    ),
    verbose=True,
    allow_delegation=False,  # This agent focuses on its specific task
    tools=[support_data_tool],  # Assign the data fetching tool
    llm=gemini_llm  # Use the configured Gemini LLM
)





# Agent 2: Process optimizer
process_optimizer = Agent(
    role='Process Optimization Specialist',
    goal='Identify bottlenecks and inefficiencies in current support processes based on the data analysis. Propose actionable improvements.',
    backstory=(
        """You are a specialist in optimizing business processes, particularly in customer support.
        You excel at pinpointing root causes of delays and inefficiencies and suggesting concrete solutions."""
    ),
    verbose=True,
    allow_delegation=False,
    # No tools needed, this agent relies on the context provided by data_analyst.
    llm=gemini_llm
)




# Agent 3: Report writer
report_writer = Agent(
    role='Executive Report Writer',
    goal='Compile the analysis and improvement suggestions into a concise, clear, and actionable report for the COO.',
    backstory=(
        """You are a skilled writer adept at creating executive summaries and reports.
        You focus on clarity, conciseness, and highlighting the most critical information and recommendations for senior leadership."""
    ),
    verbose=True,
    allow_delegation=False,
    llm=gemini_llm
)

# Tasks

In [7]:
from crewai import Task

# Task 1: Analyze data
analysis_task = Task(
    description=(
        """Fetch and analyze the latest customer support interaction data (tickets, feedback, call logs)
        focusing on the last quarter. Identify the top 3-5 recurring issues, quantify their frequency
        and impact (e.g., resolution time, customer sentiment). Use the Customer Support Data Fetcher tool."""
    ),
    expected_output=(
        """A summary report detailing the key findings from the customer support data analysis, including:
- Top 3-5 recurring issues with frequency.
- Average resolution times for these issues.
- Key customer pain points mentioned in feedback.
- Any notable trends in sentiment or support agent observations."""
    ),
    agent=data_analyst  # Assign task to the data_analyst agent
)



# Task 2: Identify bottlenecks and suggest improvements
optimization_task = Task(
    description=(
        """Based on the data analysis report provided by the Data Analyst, identify the primary bottlenecks
        in the support processes contributing to the identified issues (especially the top recurring ones).
        Propose 2-3 concrete, actionable process improvements to address these bottlenecks.
        Consider potential impact and ease of implementation."""
    ),
    expected_output=(
        """A concise list identifying the main process bottlenecks (e.g., lack of documentation for agents,
        complex escalation path, UI issues) linked to the key problems.
A list of 2-3 specific, actionable recommendations for process improvement
(e.g., update agent knowledge base, simplify password reset UI, implement proactive monitoring)."""
    ),
    agent=process_optimizer  # Assign task to the process_optimizer agent
    # This task implicitly uses the output of analysis_task as context
)


# Task 3: Compile COO report
report_task = Task(
    description=(
        """Compile the findings from the Data Analyst and the recommendations from the Process Optimization Specialist
        into a single, concise executive report for the COO. The report should clearly state:
1. The most critical customer support issues identified (with brief data points).
2. The key process bottlenecks causing these issues.
3. The recommended process improvements.
Ensure the report is easy to understand, focuses on actionable insights, and is formatted professionally."""
    ),
    expected_output=(
        """A well-structured executive report (max 1 page) summarizing the critical support issues,
        underlying process bottlenecks, and clear, actionable recommendations for the COO.
        Use clear headings and bullet points."""
    ),
    agent=report_writer  # Assign task to the report_writer agent
)

# Crew

In [8]:
from crewai import Crew, Process

support_analysis_crew = Crew(
    agents=[data_analyst, process_optimizer, report_writer],
    tasks=[analysis_task, optimization_task, report_task],
    process=Process.sequential,  # Tasks will run sequentially in the order defined
    verbose=True
)

# Run the Crew

In [9]:
import asyncio

async def run_crew():
    result = await support_analysis_crew.akickoff(inputs={'data_query': 'last quarter support data'})
    return result

# In Jupyter, top-level await works directly
result = await run_crew()
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 462529a3-a41f-46d6-8377-99e2394b6511                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Fetch and analyze the latest customer support interaction data (tickets, feedback, call logs)            │
│          focusing on the last quarter. Identify the top 3-5 recurring issues, quantify their frequency          │
│          and impact (e.g., resolution time, customer sentiment). Use the Customer Support Data Fetcher tool.    │
│  ID: dbb7dd6d-c2d5-4aea-89f8-6d2ee0cba151                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Data Analyst                                                                           │
│                                                                                                                 │
│  Task: Fetch and analyze the latest customer support interaction data (tickets, feedback, call logs)            │
│          focusing on the last quarter. Identify the top 3-5 recurring issues, quantify their frequency          │
│          and impact (e.g., resolution time, customer sentiment). Use the Customer Support Data Fetcher tool.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

--- Fetching data for query: last quarter ---

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: customer_support_data_fetcher                                                                            │
│  Args: {'argument': 'last quarter'}                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Report Writer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # EXECUTIVE REPORT                                                                                             │
│                                                                                                                 │
│  **TO:** Chief Operating Officer (COO)                                                                          │
│  **FROM:** Operations & Process Improvement Team                                                                │
│  **DATE:** October 24, 2023                                                                                     │
│  **SUBJECT:** Operational Analysis & Recommendations: Customer Support Optimization                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Executive Summary                                                                                          │
│  A recent quarterly analysis of customer support operations revealed critical inefficiencies in handling user   │
│  authentication and feedback loops. The primary operational bottleneck—**authentication and login resolution    │
│  taking an average of 48 hours**—is driving significant customer friction and inflating support costs. By       │
│  addressing agent diagnostic tool deficits, refining self-service UI workflows, and decoupling feature          │
│  requests from support queues, operations can dramatically reduce resolution times, lower ticket volume, and    │
│  improve overall customer sentiment.                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Critical Customer Support Issues & Key Data Points                                                      │
│                                                                                                                 │
│  | Rank / Category | Metrics & Frequencies | Resolution Time (MTTR) | Core Operational Risk |                   │
│  | :--- | :--- | :--- | :--- |                                                                                  │
│  | **1. Login & Auth Failures** | **50 Tickets** (Highest Volume) | **~48 Hours** | Customer downtime, rising   │
│  dissatisfaction, and excessive manual investigation. |                                                         │
│  | **2. Account Verification** | High Volume (Call Logs) | Variable | Drives excessive inbound phone support    │
│  costs due to confusing identity checks. |                                                                      │
│  | **3. Billing Discrepancies** | 30 Tickets | ~12 Hours | Stable; operating within acceptable turnaround       │
│  benchmarks. |                                                                                                  │
│  | **4. Feature Requests** | 20 Tickets | N/A (Closed u

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compile the findings from the Data Analyst and the recommendations from the Process Optimization         │
│  Specialist                                                                                                     │
│          into a single, concise executive report for the COO. The report should clearly state:                  │
│  1. The most critical customer support issues identified (with brief data points).                              │
│  2. The key process bottlenecks causing these issues.                                                           │
│  3. The recommended process improvements.                                                                       │
│  Ensure the report is easy to understand, focuses on actionable insights, and is formatted professionally.      │
│  Agent: Executive Report Writer                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 462529a3-a41f-46d6-8377-99e2394b6511                                                                       │
│  Final Output: # EXECUTIVE REPORT                                                                               │
│                                                                                                                 │
│  **TO:** Chief Operating Officer (COO)                                                                          │
│  **FROM:** Operations & Process Improvement Team                                                                │
│  **DATE:** October 24, 2023                                                                                     │
│  **SUBJECT:** Operational Analysis & Recommendations: Customer Support Optimization                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Executive Summary                                                                                          │
│  A recent quarterly analysis of customer support operations revealed critical inefficiencies in handling user   │
│  authentication and feedback loops. The primary operational bottleneck—**authentication and login resolution    │
│  taking an average of 48 hours**—is driving significant customer friction and inflating support costs. By       │
│  addressing agent diagnostic tool deficits, refining self-service UI workflows, and decoupling feature          │
│  requests from support queues, operations can dramatically reduce resolution times, lower ticket volume, and    │
│  improve overall customer sentiment.                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Critical Customer Support Issues & Key Data Points                                                      │
│                                                                                                                 │
│  | Rank / Category | Metrics & Frequencies | Resolution Time (MTTR) | Core Operational Risk |                   │
│  | :--- | :--- | :--- | :--- |                                                                                  │
│  | **1. Login & Auth Failures** | **50 Tickets** (Highest Volume) | **~48 Hours** | Customer downtime, rising   │
│  dissatisfaction, and excessive manual investigation. |                                                         │
│  | **2. Account Verification** | High Volume (Call Logs) | Variable | Drives excessive inbound phone support    │
│  costs due to confusing identity checks. |                                                                      │
│  | **3. Billing Discrepancies** | 30 Tickets | ~12 Hours | Stable; operating within acceptable turnaround       │
│  benchmarks. |                                                                                                  │
│  | **4. Feature Requests** | 20 Tickets | N/A (Closed 

# EXECUTIVE REPORT

**TO:** Chief Operating Officer (COO)  
**FROM:** Operations & Process Improvement Team  
**DATE:** October 24, 2023  
**SUBJECT:** Operational Analysis & Recommendations: Customer Support Optimization  

---

### Executive Summary
A recent quarterly analysis of customer support operations revealed critical inefficiencies in handling user authentication and feedback loops. The primary operational bottleneck—**authentication and login resolution taking an average of 48 hours**—is driving significant customer friction and inflating support costs. By addressing agent diagnostic tool deficits, refining self-service UI workflows, and decoupling feature requests from support queues, operations can dramatically reduce resolution times, lower ticket volume, and improve overall customer sentiment.

---

### 1. Critical Customer Support Issues & Key Data Points

| Rank / Category | Metrics & Frequencies | Resolution Time (MTTR) | Core Operational Risk |
| :--- | :--- | :--- |

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
from IPython.display import Markdown
Markdown(result.raw)

# EXECUTIVE REPORT

**TO:** Chief Operating Officer (COO)  
**FROM:** Operations & Process Improvement Team  
**DATE:** October 24, 2023  
**SUBJECT:** Operational Analysis & Recommendations: Customer Support Optimization  

---

### Executive Summary
A recent quarterly analysis of customer support operations revealed critical inefficiencies in handling user authentication and feedback loops. The primary operational bottleneck—**authentication and login resolution taking an average of 48 hours**—is driving significant customer friction and inflating support costs. By addressing agent diagnostic tool deficits, refining self-service UI workflows, and decoupling feature requests from support queues, operations can dramatically reduce resolution times, lower ticket volume, and improve overall customer sentiment.

---

### 1. Critical Customer Support Issues & Key Data Points

| Rank / Category | Metrics & Frequencies | Resolution Time (MTTR) | Core Operational Risk |
| :--- | :--- | :--- | :--- |
| **1. Login & Auth Failures** | **50 Tickets** (Highest Volume) | **~48 Hours** | Customer downtime, rising dissatisfaction, and excessive manual investigation. |
| **2. Account Verification** | High Volume (Call Logs) | Variable | Drives excessive inbound phone support costs due to confusing identity checks. |
| **3. Billing Discrepancies** | 30 Tickets | ~12 Hours | Stable; operating within acceptable turnaround benchmarks. |
| **4. Feature Requests** | 20 Tickets | N/A (Closed unresolved) | Systematically closed without customer follow-up, eroding user trust. |

---

### 2. Key Process Bottlenecks (Root Causes)

1. **Diagnostic Tool Deficit (Login Issues):** Support agents lack session-replay capabilities (e.g., user-side console logs). Agents spend excessive back-and-forth time attempting to manually reproduce obscure authentication bugs, causing the extended 48-hour resolution window.
2. **Friction-Heavy Self-Service Architecture:** Self-service password reset and verification workflows suffer from confusing UI design and vague error messages. Users default to high-cost support channels (phone/tickets) rather than resolving issues independently.
3. **Broken Product-Support Feedback Loop:** Support agents use ticket closure as a default mechanism for feature requests because no formal handoff pipeline exists between Support and Product Management.

---

### 3. Actionable Process Improvement Recommendations

#### Priority 1: Deploy Session Telemetry Tools & Standardized Triage Workflows
* **Action:** Integrate real-time session replay/logging tools (e.g., LogRocket, FullStory) directly into the agent dashboard, paired with a standardized 3-step triage script for Tier 1 agents.
* **Impact:** **HIGH** — Cuts average login resolution time from **48 hours to under 4 hours**.
* **Ease of Implementation:** **Medium** (Requires third-party API integration with current support tools).

#### Priority 2: Redesign Password Recovery & Verification UI/UX
* **Action:** Redesign the self-service password reset flow with inline micro-copy, clear error feedback, and simplified verification steps.
* **Impact:** **HIGH** — Deflects **40% to 50%** of inbound authentication tickets and phone calls.
* **Ease of Implementation:** **Medium** (Requires cross-functional effort between UX and Frontend engineering).

#### Priority 3: Transition Feature Requests to a Dedicated Feedback Pipeline
* **Action:** Decouple feature requests from the support queue by launching an automated customer feedback portal (e.g., Canny or Productboard) integrated with Product Management.
* **Impact:** **MEDIUM** — Eliminates non-support ticket clutter and automates status updates to customers.
* **Ease of Implementation:** **High / Easy** (Off-the-shelf software with minimal technical setup).

---

### 4. Strategic Next Steps for Executive Sign-off

1. **Authorize Engineering Resource Allocation:** Grant approval for 2–3 sprint cycles of UX/Engineering effort to deploy telemetry tools (Priority 1) and update recovery UI (Priority 2).
2. **Product Management Alignment:** Direct Product Leadership to adopt a feedback portal software to absorb feature request management by next month.